In [ ]:
import torch
from transformers import pipeline, BitsAndBytesConfig
import requests
from PIL import Image
import pandas as pd
import numpy as np




df = pd.read_excel('./content_simulation_test_company.xlsx')
# Assuming df is your DataFrame containing the 'image' column
# Fetching the first row's 'image' value

import swifter
import re

def extract_link(x):
    match_ = re.search(r"previewUrl='(.*?)'|thumbnailUrl='(.*?)'", x)
    if match_:
        return match_.group(1) or match_.group(2)
    else:
        return "URL not found."
    
df['extracted_links'] = df['media'].swifter.apply(extract_link)

df_1 = df.head(1000)

quantization_config = BitsAndBytesConfig(
         load_in_4bit=True,
         bnb_4bit_compute_dtype=torch.float16
    )

model_id = "llava-hf/llava-1.5-7b-hf"
max_new_tokens = 200
pipe = pipeline("image-to-text", model=model_id, model_kwargs={"quantization_config": quantization_config})



examples = [
    ["image: df['extracted_links].iloc[0]", "Date: 2020-12-12 00:47:00, brand_name: tim hortons, username: TimHortonsPH, likes: 1", "outputs: Spend your weekend morning with a Ham, Egg, and Cheese Wrap paired with a sweet Iced French Vanilla! ☀️ Order yours now via dine-in, takeout, and delivery. #TimHortonsPH <hyperlink>"],
    ["image: df['extracted_links].iloc[13]", "Date: 2018-08-17 13:00:18, brand_name: pfizer, username: Pfizer_UK, likes: 2","outputs:#DYK whilst #TickBorneEncephalitis is not found in the UK, it can occur in many popular holiday destinations in Europe. Find out more: <hyperlink> #TickAlert #TravelBug <hyperlink>"]]
prompt_examples = "\n".join([f"{example}" for example in examples])
arr = []
def tweet(link, brand, username, likes,date,arr):
    first_image_url = link
    
    try:
        # Attempt to load the image from the URL
        image = Image.open(requests.get(first_image_url, stream=True).raw)
    except Exception as e:
        print(f"Error loading image from {first_image_url}: {e}")
        return "Image not loaded"
    
    brand_name = brand
    username = username
    likes = likes 
    date = date 
    prompt2 = f"USER: <image>\nCreate an engaging tweet caption for this image that aligns with {brand_name} style. Do consider the other name {username} . the date of the tweet is {date} , see if there is anything special on this day. Identify the place and text in the image. Also identify the objects in the image. Craft it to gather {likes} likes. Avoid direct brand mentions or like counts in the output. Incorporate relevant hashtags for maximum engagement."
    prompt = f"USER: <image>\nGenerate a tweet caption for this specific image that has brand name = {brand_name}, make a marketing tweet text in a way that it receives {likes} likes.the date of the tweet is {date}. Do not print the brand name and likes, and also provide hashtags."
    

    outputs = pipe(image, prompt=prompt, generate_kwargs={"max_new_tokens": 200})
    text = outputs[0]['generated_text']
    original_string = text
    sentence_to_remove = "USER:this is the input image:  \nCreate a compelling tweet caption for the image featuring the brand \'pepsico\' and has twitter username \'PepsiCoAMESA\'. identify the text in the image and Craft a tweet such a way to get \'1\' likes. Avoid mentioning the likes explicitly. Include relevant hashtags to enhance visibility and appeal. Be creative and engaging in your description. Also try identifying the place in the image and also the objects present in the image. mention the place in the image. Do mention the username . Give one liner text only. \n\n"
    result_string = original_string.replace(sentence_to_remove, "")
    print(result_string)
    arr.append(result_string)
    arr_1 = pd.DataFrame(arr)
    file_path = 'df_10K_test_company.csv'
    arr_1.to_csv(file_path, index=False)
    return arr

    


def build(df):
    length = len(df)
    df_re = pd.read_csv('./df_10K_test_company.csv')
    arr = df_re.iloc[:,0].tolist()
    for i in range(length):
        arr = tweet(df['extracted_links'].iloc[i],df['inferred company'].iloc[i],df['username'].iloc[i],df['likes'].iloc[i],df['date'].iloc[i],arr)
        print(i)




build(df.iloc[87:110])
#image_2  = Image.open(requests.get(img_2_url, stream=True).raw)










#description = df.loc[0, 'description']




#prompt = f"USER: You are the expert tweet content creator! Use this <image>\nGenerate a one line tweet \
#caption for this specific image that has brand name = {brand_name}, \
#make a marketing tweet text in a way that it receives {likes} likes \
#and do mention {username}.Try identifying the place. Identify the \
#objects in the image. Identify the text in the image. Consider all \
#this and give an amazing tweet. Do not print likes, and also provide \
#hashtags and emojis. add a string at the end <hyperlink><hyperlink> if \
#you feel so"

 





 




# Save the DataFrame to a CSV file



